In [2]:
from langchain.chat_models import init_chat_model
import os
import subprocess

from dotenv import load_dotenv
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage
from langchain_core.tools import tool

load_dotenv(override=True)

os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_MODEL"] = os.getenv("OPENAI_MODEL")

SYSTEM = f"You are a coding agent at {os.getcwd()}. Use bash to solve tasks. Act, don't explain."


@tool
def bash(command: str) -> str:
    """
    Run a shell command.
    """
    dangerous = ["rm -rf /", "sudo", "shutdown", "reboot", "> /dev/"]
    if any(d in command for d in dangerous):
        return "Error: Dangerous command blocked"
    try:
        r = subprocess.run(command, shell=True, cwd=os.getcwd(),
                           capture_output=True, text=True, timeout=120)
        out = (r.stdout + r.stderr).strip()
        result = out[:50000] if out else "(no output)"
        return result
    except subprocess.TimeoutExpired:
        return "Error: Timeout (120s)"
    except (FileNotFoundError, OSError) as e:
        return f"Error: {e}"

tools = [bash]

tool_dict = {tool.name: tool for tool in tools}

model = init_chat_model(
    model=os.getenv("OPENAI_MODEL"),  # 模型名称，这里选择qwen3.5-plus，这是一个多模态模型，支持图片、文本、音频、视频
    model_provider="openai",
    base_url=os.getenv("OPENAI_BASE_URL"),
    api_key=os.getenv("OPENAI_API_KEY")
)

model_with_tools = model.bind_tools(tools)


C:\Software\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [11]:

def agent_loop(messages: list):
    while True:
        response = model_with_tools.invoke(messages)

        messages.append(response)

        if (len(response.tool_calls) == 0):
            return

        for tool_call in response.tool_calls:
            tool = tool_dict[tool_call["name"]]
            tool_response = tool.invoke(tool_call)
            print(tool_response.content)
            # messages.append(tool_response)
            messages.append(tool_response)



In [12]:
messages = [HumanMessage("我想看当前目录!")]
agent_loop(messages)
messages

'ls' 不是内部或外部命令，也不是可运行的程序
或批处理文件。
驱动器 C 中的卷是 Windows-SSD
 卷的序列号是 B0CD-6E76

 C:\Project\PycharmProject\learn-claude-code\s01_agent_loop 的目录

2026/05/27  15:23    <DIR>          .
2026/05/27  10:19    <DIR>          ..
2026/05/27  10:06             4,877 code.py
2026/05/27  10:06    <DIR>          images
2026/05/27  15:23            15,432 langchain_code.ipynb
2026/05/27  15:22             3,695 langchain_code.py
2026/05/27  10:06             8,486 README.en.md
2026/05/27  10:06             9,889 README.ja.md
2026/05/27  10:06             7,917 README.md
2026/05/27  14:43    <DIR>          __pycache__
               6 个文件         50,296 字节
               4 个目录 163,071,184,896 可用字节


[HumanMessage(content='我想看当前目录!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 47, 'total_tokens': 64, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 55, 'engine_ttft_ms': 163, 'engine_ttlt_ms': 1092, 'pre_inference_ms': 95, 'service_tbt_ms': 57, 'service_ttft_ms': 474, 'service_ttlt_ms': 1404, 'total_duration_ms': 1354, 'user_visible_ttft_ms': 379}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-Dk2u9W60EHOHiCb1P3vUdd3fEISre', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e6851-618b-7521-9f57-04b617a1fb7e-0', tool_calls=[{'name': 'bash

In [10]:
messages[-1].content
#
# for message in messages:
#     message.pretty_print()

'你好! 有什么我可以帮助你的吗?'

In [56]:
messages = [HumanMessage("帮我查询当前目录文件有哪些!")]
agent_loop(messages)

In [57]:
for message in messages:
    message.pretty_print()

================================ Human Message =================================

帮我查询当前目录文件有哪些!
================================== Ai Message ==================================
Tool Calls:
  bash (call_PIicjjbm5mY2VWs4yjAU6yhQ)
 Call ID: call_PIicjjbm5mY2VWs4yjAU6yhQ
  Args:
    command: ls -l
================================= Tool Message =================================
Name: bash

'ls' 不是内部或外部命令，也不是可运行的程序
或批处理文件。
================================== Ai Message ==================================
Tool Calls:
  bash (call_DNSZvb6u5cd4BpbJFl2JFMLL)
 Call ID: call_DNSZvb6u5cd4BpbJFl2JFMLL
  Args:
    command: dir
================================= Tool Message =================================
Name: bash

驱动器 C 中的卷是 Windows-SSD
 卷的序列号是 B0CD-6E76

 C:\Project\PycharmProject\learn-claude-code\s01_agent_loop 的目录

2026/05/27  11:37    <DIR>          .
2026/05/27  10:19    <DIR>          ..
2026/05/27  10:06             4,877 code.py
2026/05/27  10:06    <DIR>          images
2026/05/27  11:37

In [58]:
messages

[HumanMessage(content='帮我查询当前目录文件有哪些!', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 17, 'prompt_tokens': 49, 'total_tokens': 66, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}, 'latency_checkpoint': {'engine_tbt_ms': 41, 'engine_ttft_ms': 109, 'engine_ttlt_ms': 806, 'pre_inference_ms': 125, 'service_tbt_ms': 41, 'service_ttft_ms': 637, 'service_ttlt_ms': 1321, 'total_duration_ms': 1209, 'user_visible_ttft_ms': 512}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_eb37e061ec', 'id': 'chatcmpl-DjzN8q27O8ycIIrq7LyMiULe96Ym7', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019e6782-21f1-75b0-92a8-d0a43aec4dac-0', tool_calls=[{'name':